# DataPy - Übung (Woche 5)

**pandas, Web Scraping und REST APIs** | WiSe26/27 | Prof. Dr. Selcan Ipek-Ugay

---

## Ziel der Übung

Ihr festigt den Stoff aus der Vorlesung: **Series und DataFrames**, **Indexing mit `loc`**, **Boolean Indexing**, **fehlende Werte**, **Group-by**, **Joins**, **HTML mit Beautiful Soup und `read_html` auslesen**, **HTTP-Statuscodes** und **Antworten einer REST API** verarbeiten. In Teil D wertet ihr **Luftqualitätsdaten** aus.

## Vorbereitung

Führt diese Zelle **einmal** aus. Sie importiert die Pakete und wird **nicht verändert**. Fehlt ein Paket, installiert es im Terminal mit `pip install pandas lxml html5lib beautifulsoup4 requests`.

In [39]:
from io import StringIO

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

---

# Teil A: Series

Kleine Funktionen rund um **Erzeugen, Indexing, Boolean Indexing, `loc`, fehlende Werte und das Rechnen über den Index**. Alle Funktionen **geben ein Ergebnis zurück** und verändern ihre Eingabe nicht.

### Aufgabe A1: Series erzeugen

Schreibt `temperature_series(values)`: Die Funktion bekommt eine Liste mit **fünf Temperaturen** und gibt eine **Series** zurück, deren **Index** die Wochentage `'Mo'`, `'Di'`, `'Mi'`, `'Do'`, `'Fr'` sind und die den **Namen** `'temp'` trägt.

Beispiel: `temperature_series([3, 5, 8, 6, 4])` hat am Mittwoch den Wert `8`.

In [40]:
def temperature_series(values: list) -> pd.Series:
    """Gibt eine Series mit den Wochentagen Mo bis Fr als Index und dem Namen 'temp' zurück."""
    # TODO
    wd = ['Mo', 'Di', 'Mi', 'Do', 'Fr']
    result = pd.Series(data=values, index=wd)
    print(result)
    result.name = "temp"
    return result

In [41]:
s = temperature_series([3, 5, 8, 6, 4])
assert isinstance(s, pd.Series), "Es soll eine pandas Series zurückgegeben werden"
assert s.index.tolist() == ['Mo', 'Di', 'Mi', 'Do', 'Fr'], f"Index: {s.index.tolist()}"
assert s.loc['Mi'] == 8, "Mittwoch"
assert s.name == 'temp', f"Name: {s.name}"
assert s.tolist() == [3, 5, 8, 6, 4], "Werte"
print("✓ Aufgabe A1 bestanden")

Mo    3
Di    5
Mi    8
Do    6
Fr    4
dtype: int64
✓ Aufgabe A1 bestanden


### Aufgabe A2: Boolean Indexing

Schreibt `warm_days(s, limit)`: Die Funktion gibt nur die Einträge der Series `s` zurück, deren Wert **größer als `limit`** ist (**Boolean Indexing**). Der Index bleibt erhalten, `s` wird nicht verändert.

Beispiel: Für `[3, 5, 8, 6, 4]` und `limit = 4` bleiben `Di`, `Mi` und `Do` übrig.

In [42]:
def warm_days(s: pd.Series, limit: float) -> pd.Series:
    """Gibt alle Einträge von s zurück, die größer als limit sind."""
    # TODO
    result = s[s > limit]
    print(result)
    return result 

In [43]:
s = pd.Series([3, 5, 8, 6, 4], index=['Mo', 'Di', 'Mi', 'Do', 'Fr'])
result = warm_days(s, 4)
assert isinstance(result, pd.Series), "Es soll eine Series zurückgegeben werden"
assert result.index.tolist() == ['Di', 'Mi', 'Do'], f"Index: {result.index.tolist()}"
assert result.tolist() == [5, 8, 6], f"Werte: {result.tolist()}"
assert len(s) == 5, "Das Original darf nicht verändert werden"
assert warm_days(s, 100).empty, "Kein Eintrag über 100"
print("✓ Aufgabe A2 bestanden")

Di    5
Mi    8
Do    6
dtype: int64
Series([], dtype: int64)
✓ Aufgabe A2 bestanden


### Aufgabe A3: Label-Slicing mit `loc`

Schreibt `slice_by_label(s, start, end)`: Die Funktion gibt die Einträge von `start` **bis einschließlich `end`** zurück. Verwendet `loc`.

*Erinnert euch: Beim Slicing mit **Labels** ist der Endpunkt **eingeschlossen**, bei Positionen (`iloc`) nicht.*

Beispiel: `slice_by_label(s, 'Di', 'Do')` liefert die drei Tage `Di`, `Mi` und `Do`.

In [44]:
def slice_by_label(s: pd.Series, start: str, end: str) -> pd.Series:
    """Gibt die Einträge von start bis einschließlich end zurück."""
    # TODO
    print("s:\n",s," \nstart: ",start," \nend: ", end )
    data = s[start:end]
    print("data:\n",data)
    return data

In [45]:
s = pd.Series([3, 5, 8, 6, 4], index=['Mo', 'Di', 'Mi', 'Do', 'Fr'])
result = slice_by_label(s, 'Di', 'Do')
assert result.index.tolist() == ['Di', 'Mi', 'Do'], f"Index: {result.index.tolist()}"
assert result.tolist() == [5, 8, 6], f"Werte: {result.tolist()}"
assert slice_by_label(s, 'Mo', 'Mo').tolist() == [3], "Ein einzelner Eintrag"
assert slice_by_label(s, 'Mi', 'Fr').index.tolist() == ['Mi', 'Do', 'Fr'], "Bis zum Ende"
print("✓ Aufgabe A3 bestanden")

s:
 Mo    3
Di    5
Mi    8
Do    6
Fr    4
dtype: int64  
start:  Di  
end:  Do
data:
 Di    5
Mi    8
Do    6
dtype: int64
s:
 Mo    3
Di    5
Mi    8
Do    6
Fr    4
dtype: int64  
start:  Mo  
end:  Mo
data:
 Mo    3
dtype: int64
s:
 Mo    3
Di    5
Mi    8
Do    6
Fr    4
dtype: int64  
start:  Mi  
end:  Fr
data:
 Mi    8
Do    6
Fr    4
dtype: int64
✓ Aufgabe A3 bestanden


### Aufgabe A4: Fehlende Werte behandeln

Schreibt `clean_missing(s, how)`: Die Funktion gibt eine **neue** Series ohne fehlende Werte zurück. Der Parameter `how` legt fest, wie:

- `'drop'`: Einträge mit fehlenden Werten **löschen** (`dropna`)
- `'zero'`: fehlende Werte durch `0` **ersetzen** (`fillna`)
- `'mean'`: fehlende Werte durch den **Mittelwert** der vorhandenen Werte ersetzen

Bei jedem anderen Wert für `how` wird ein `ValueError` ausgelöst (`raise ValueError(...)`).

*Frage zum Nachdenken: Welche Variante verändert den Mittelwert der Series, welche nicht?*

In [46]:
def clean_missing(s: pd.Series, how: str) -> pd.Series:
    """Entfernt oder ersetzt fehlende Werte ('drop', 'zero' oder 'mean')."""
    # TODO
    if how == "drop":
        return s.dropna()
    elif how == "zero":
        return s.fillna(0)
    elif how == "mean":
        return s.fillna(s.mean())
    else:
        raise ValueError(f"{how} ist kein korrekter Wert für 'how'")

In [47]:
s = pd.Series([2.0, np.nan, 4.0, np.nan, 6.0], index=list('abcde'))
dropped = clean_missing(s, 'drop')
assert dropped.index.tolist() == ['a', 'c', 'e'] and dropped.tolist() == [2.0, 4.0, 6.0], f"drop: {dropped.tolist()}"
assert clean_missing(s, 'zero').tolist() == [2.0, 0.0, 4.0, 0.0, 6.0], "zero"
assert clean_missing(s, 'mean').tolist() == [2.0, 4.0, 4.0, 4.0, 6.0], "mean"
skew = pd.Series([1.0, np.nan, 2.0, 9.0])
assert clean_missing(skew, 'mean').tolist() == [1.0, 4.0, 2.0, 9.0], "mean (nicht der Median)"
assert int(s.isna().sum()) == 2, "Das Original darf nicht verändert werden"
try:
    clean_missing(s, 'median')
    raise AssertionError("Bei einer unbekannten Methode soll ein ValueError ausgelöst werden")
except ValueError:
    pass
print("✓ Aufgabe A4 bestanden")

✓ Aufgabe A4 bestanden


### Aufgabe A5: Rechnen über den Index

Schreibt `add_stocks(a, b)`: Die Funktion **addiert** die beiden Series `a` und `b`. Dabei werden die Werte über den **Index** zugeordnet (nicht über die Position). Kommt ein Eintrag nur in **einer** Series vor, zählt er in der anderen als `0`, das Ergebnis darf also **kein `NaN`** enthalten.

*Tipp: Die Methode `Series.add` hat den Parameter `fill_value`.*

Beispiel: Hat `a` `apples = 1` und `b` `apples = 3`, ist das Ergebnis `4`. Kommt `lemons` nur in `b` mit `5` vor, ist das Ergebnis `5`.

In [48]:
def add_stocks(a: pd.Series, b: pd.Series) -> pd.Series:
    """Addiert zwei Series über den Index, fehlende Einträge zählen als 0."""
    # TODO
    return a.add(b, fill_value=0)

In [49]:
a = pd.Series([1, 0, 2], index=['apples', 'oranges', 'bananas'])
b = pd.Series([1, 5, 3], index=['bananas', 'lemons', 'apples'])
result = add_stocks(a, b)
assert not result.isna().any(), "Das Ergebnis darf kein NaN enthalten"
assert result.to_dict() == {'apples': 4, 'bananas': 3, 'lemons': 5, 'oranges': 0}, f"Ergebnis: {result.to_dict()}"
assert a.tolist() == [1, 0, 2] and b.tolist() == [1, 5, 3], "Die Originale dürfen nicht verändert werden"
print("✓ Aufgabe A5 bestanden")

✓ Aufgabe A5 bestanden


---

# Teil B: DataFrames, Group-by und Joins

Hier arbeitet ihr mit **Tabellen**: erzeugen, filtern, sortieren, neue Spalten berechnen, **gruppieren** und **verbinden**.

### Aufgabe B1: DataFrame erzeugen

Schreibt `build_students(names, semesters, grades)`: Die Funktion gibt einen **DataFrame** mit den drei Spalten `name`, `semester` und `note` (in dieser Reihenfolge) zurück, eine Zeile pro Student:in.

Beispiel: `build_students(['Ada', 'Ben'], [1, 1], [2.3, 3.0])` hat zwei Zeilen und drei Spalten.

In [50]:
def build_students(names: list, semesters: list, grades: list) -> pd.DataFrame:
    """Gibt einen DataFrame mit den Spalten name, semester und note zurück."""
    # TODO
    pass

In [51]:
df = build_students(['Ada', 'Ben', 'Cem'], [1, 1, 2], [2.3, 3.0, 1.3])
assert isinstance(df, pd.DataFrame), "Es soll ein DataFrame zurückgegeben werden"
assert df.columns.tolist() == ['name', 'semester', 'note'], f"Spalten: {df.columns.tolist()}"
assert df.shape == (3, 3), f"Form: {df.shape}"
assert df.loc[2, 'name'] == 'Cem', "Zugriff mit loc"
assert df['note'].tolist() == [2.3, 3.0, 1.3], "Spalte note"
print("✓ Aufgabe B1 bestanden")

AssertionError: Es soll ein DataFrame zurückgegeben werden

### Aufgabe B2: Filtern und sortieren

Schreibt `good_students(df, limit)`: Die Funktion gibt alle Zeilen zurück, deren `note` **kleiner als `limit`** ist, **aufsteigend nach `note` sortiert** (beste Note zuerst). Der Index des Ergebnisses soll wieder bei `0` beginnen (`reset_index(drop=True)`). Der ursprüngliche DataFrame bleibt unverändert.

In [ ]:
def good_students(df: pd.DataFrame, limit: float) -> pd.DataFrame:
    """Gibt alle Zeilen mit note < limit zurück, sortiert nach note, mit neuem Index."""
    # TODO
    pass

In [ ]:
df = build_students(['Ada', 'Ben', 'Cem', 'Dina', 'Eva'], [1, 1, 2, 2, 2], [2.3, 3.0, 1.3, 4.7, 2.0])
result = good_students(df, 2.5)
assert result['name'].tolist() == ['Cem', 'Eva', 'Ada'], f"Namen: {result['name'].tolist()}"
assert result['note'].tolist() == [1.3, 2.0, 2.3], "Noten sortiert"
assert result.index.tolist() == [0, 1, 2], f"Index: {result.index.tolist()}"
assert df['name'].tolist() == ['Ada', 'Ben', 'Cem', 'Dina', 'Eva'], "Das Original darf nicht verändert werden"
assert good_students(df, 2.0).shape[0] == 1, "note < 2.0 (kleiner, nicht kleiner gleich)"
print("✓ Aufgabe B2 bestanden")

### Aufgabe B3: Neue Spalte berechnen

Schreibt `add_passed(df)`: Die Funktion gibt eine **Kopie** des DataFrames mit einer zusätzlichen Spalte `bestanden` zurück. Sie ist `True`, wenn die `note` **höchstens `4.0`** beträgt, sonst `False`. Das Original bleibt unverändert (es bekommt keine neue Spalte).

In [ ]:
def add_passed(df: pd.DataFrame) -> pd.DataFrame:
    """Gibt eine Kopie von df mit der Spalte bestanden (note <= 4.0) zurück."""
    # TODO
    pass

In [ ]:
df = build_students(['Ada', 'Ben', 'Cem', 'Dina', 'Eva'], [1, 1, 2, 2, 2], [2.3, 3.0, 1.3, 4.7, 2.0])
result = add_passed(df)
assert 'bestanden' in result.columns, "Die Spalte 'bestanden' fehlt"
assert result['bestanden'].tolist() == [True, True, True, False, True], f"Werte: {result['bestanden'].tolist()}"
assert result['bestanden'].dtype == bool, "Die Spalte soll Wahrheitswerte enthalten"
assert 'bestanden' not in df.columns, "Das Original darf nicht verändert werden (copy() verwenden)"
assert add_passed(build_students(['X'], [1], [4.0]))['bestanden'].tolist() == [True], "4.0 gilt als bestanden"
print("✓ Aufgabe B3 bestanden")

### Aufgabe B4: Group-by

Schreibt `mean_by_semester(df)`: Die Funktion berechnet die **mittlere Note pro Semester** (`groupby`) und gibt eine **Series** zurück, deren Index die Semester sind (aufsteigend) und deren Werte die mittleren Noten.

In [ ]:
def mean_by_semester(df: pd.DataFrame) -> pd.Series:
    """Gibt die mittlere Note je Semester als Series zurück."""
    # TODO
    pass

In [ ]:
df = build_students(['Ada', 'Ben', 'Cem', 'Dina', 'Eva'], [1, 1, 2, 2, 2], [2.3, 3.0, 1.3, 4.7, 2.0])
result = mean_by_semester(df)
assert isinstance(result, pd.Series), "Es soll eine Series zurückgegeben werden"
assert result.index.tolist() == [1, 2], f"Index: {result.index.tolist()}"
assert np.allclose(result.values, [2.65, 8 / 3]), f"Werte: {result.values}"
assert df.shape == (5, 3), "Das Original darf nicht verändert werden"
print("✓ Aufgabe B4 bestanden")

### Aufgabe B5: Joins im Vergleich

Schreibt `join_sizes(left, right)`: Die Funktion verbindet die beiden DataFrames über die gemeinsame Spalte `id` mit **`inner`**, **`left`** und **`outer`** (`pd.merge`) und gibt ein Dictionary zurück, das für jeden Join-Typ die **Anzahl der Zeilen** des Ergebnisses enthält, z. B. `{'inner': 2, 'left': 4, 'outer': 5}`.

*Frage zum Nachdenken: Was passiert, wenn ein Key in einer Tabelle mehrfach vorkommt?*

In [ ]:
def join_sizes(left: pd.DataFrame, right: pd.DataFrame) -> dict:
    """Gibt die Zeilenanzahl von inner, left und outer Join über die Spalte id zurück."""
    # TODO
    pass

In [ ]:
left = pd.DataFrame({'id': [1, 2, 3, 4], 'x': list('abcd')})
right = pd.DataFrame({'id': [3, 4, 5], 'y': [30, 40, 50]})
assert join_sizes(left, right) == {'inner': 2, 'left': 4, 'outer': 5}, f"Ergebnis: {join_sizes(left, right)}"
left2 = pd.DataFrame({'id': [1, 1, 2], 'x': list('abc')})
right2 = pd.DataFrame({'id': [1, 2], 'y': [10, 20]})
assert join_sizes(left2, right2) == {'inner': 3, 'left': 3, 'outer': 3}, "Mehrfach vorkommender Key"
assert join_sizes(left, pd.DataFrame({'id': [9], 'y': [0]})) == {'inner': 0, 'left': 4, 'outer': 5}, "Keine Übereinstimmung"
print("✓ Aufgabe B5 bestanden")

---

# Teil C: Daten aus dem Web (ohne Internet)

Damit die Übung überall läuft, arbeitet ihr hier mit **HTML-Text und Statuscodes**, die direkt im Notebook stehen. Die Techniken sind dieselben wie bei echten Webseiten.

### Aufgabe C1: HTTP-Statuscodes

Schreibt `check_status(code)`: Die Funktion ordnet einen HTTP-Statuscode seiner **Klasse** zu und gibt den passenden Text zurück:

| Codes | Text |
|---|---|
| 100 bis 199 | `'Information'` |
| 200 bis 299 | `'Erfolg'` |
| 300 bis 399 | `'Umleitung'` |
| 400 bis 499 | `'Client-Fehler'` |
| 500 bis 599 | `'Server-Fehler'` |
| alles andere | `'Unbekannt'` |

Beispiele: `check_status(200)` → `'Erfolg'`, `check_status(404)` → `'Client-Fehler'`.

In [ ]:
def check_status(code: int) -> str:
    """Ordnet einen HTTP-Statuscode seiner Klasse zu."""
    # TODO
    if code >= 100 and code <= 199:
        return "Information"
    if 200 <= code <= 299:
        return "Erfolg"
    if 300 <= code <= 399:
        return "Umleitung"
    if 400 <= code <= 499:
        return "Client-Fehler"
    if 500 <= code <= 599:
        return "Server-Fehler"
    return "Unbekannt"

In [ ]:
assert check_status(200) == 'Erfolg', "200"
assert check_status(404) == 'Client-Fehler', "404"
assert check_status(503) == 'Server-Fehler', "503"
assert check_status(301) == 'Umleitung', "301"
assert check_status(100) == 'Information', "100"
assert check_status(299) == 'Erfolg' and check_status(300) == 'Umleitung', "Klassengrenzen"
assert check_status(199) == 'Information' and check_status(599) == 'Server-Fehler', "Obere Grenzen"
assert check_status(99) == 'Unbekannt' and check_status(600) == 'Unbekannt', "Außerhalb von 100 bis 599"
print("✓ Aufgabe C1 bestanden")

✓ Aufgabe C1 bestanden


### Aufgabe C2: Links mit Beautiful Soup auslesen

Schreibt `extract_links(html)`: Die Funktion bekommt den **HTML-Text** einer Seite und gibt eine **Liste von Tupeln** `(Linktext, Ziel)` zurück, in der Reihenfolge, in der die Links im Dokument stehen. Der Linktext hat **keine Leerzeichen am Rand** (`strip()`). Links **ohne `href`** werden ignoriert.

Verwendet `BeautifulSoup(html, 'html.parser')` und `find_all`.

Beispiel: `<a href="/studium">Studium</a>` ergibt `('Studium', '/studium')`.

In [ ]:
from bs4 import BeautifulSoup

def extract_links(html: str) -> list:
    """Gibt eine Liste von (Linktext, href) für alle Links mit href zurück."""
    # TODO
    soup = BeautifulSoup(html, "html.parser")
    result = []

    for a in soup.find_all("a"):
        href = a.get("href")
        if(href is None):
            continue
        text = a.get_text().strip()
        
        result.append((text, href))

    return result

In [ ]:
html = """
<html><body>
  <p>Mehr Infos bei der <a href="https://www.bht-berlin.de">BHT</a>.</p>
  <ul>
    <li><a href="/studium">Studium</a></li>
    <li><a name="anker">Ohne Ziel</a></li>
    <li><a href="mailto:info@example.org"> Kontakt </a></li>
  </ul>
</body></html>
"""
links = extract_links(html)
assert links == [('BHT', 'https://www.bht-berlin.de'), ('Studium', '/studium'), ('Kontakt', 'mailto:info@example.org')], f"Ergebnis: {links}"
assert extract_links("<p>Kein Link</p>") == [], "Seite ohne Links"
print("✓ Aufgabe C2 bestanden")

[('BHT', 'https://www.bht-berlin.de'), ('Studium', '/studium'), ('Kontakt', 'mailto:info@example.org')]
[]
✓ Aufgabe C2 bestanden


### Aufgabe C3: Tabellen mit `read_html`

Schreibt `table_to_df(html)`: Die Funktion liest mit `pd.read_html` die **erste Tabelle** aus dem HTML-Text und gibt sie als DataFrame zurück.

*Tipp: `pd.read_html` erwartet statt eines Strings besser ein Dateiobjekt: `StringIO(html)`. Das Ergebnis ist eine **Liste** aller Tabellen der Seite.*

In [58]:
def table_to_df(html: str) -> pd.DataFrame:
    """Gibt die erste Tabelle des HTML-Texts als DataFrame zurück."""
    # TODO
    return pd.read_html(StringIO(html))[0]

In [59]:
html = """
<html><body>
  <table>
    <tr><th>Bezirk</th><th>Einwohner</th></tr>
    <tr><td>Mitte</td><td>332100</td></tr>
    <tr><td>Pankow</td><td>368956</td></tr>
  </table>
  <table>
    <tr><th>Anderes</th></tr>
    <tr><td>x</td></tr>
  </table>
</body></html>
"""
table = table_to_df(html)
assert isinstance(table, pd.DataFrame), "Es soll ein DataFrame zurückgegeben werden (keine Liste)"
assert table.columns.tolist() == ['Bezirk', 'Einwohner'], f"Spalten: {table.columns.tolist()}"
assert table.shape == (2, 2), f"Form: {table.shape}"
assert table['Einwohner'].tolist() == [332100, 368956], "Werte"
print("✓ Aufgabe C3 bestanden")

ImportError: `Import lxml` failed.  Use pip, conda, or your preferred package management tool to install the lxml package.

---

# Teil D: Hauptaufgabe "Luftqualität"

Mehrere Messstationen liefern ihre Daten über eine **REST API** als JSON. Ihr wertet die Antworten Schritt für Schritt aus und gebt einen Bericht aus. Die Antworten stehen als Python-Dictionaries im Notebook, ihr braucht also **keinen Token und kein Internet**.

Die fünf Schritte **bauen aufeinander auf**: `responses_to_df` nutzt `parse_response`, `add_category` nutzt `aqi_category`, `report` nutzt `add_category`.

### Aufgabe D1: Eine API-Antwort auswerten

Die Luftqualitäts-API (AQICN) liefert ihre Antwort als **JSON**, in Python also als verschachteltes **Dictionary**, z. B.:

```python
{"status": "ok",
 "data": {"aqi": 42, "city": {"name": "Berlin Mitte"}, "time": {"s": "2026-10-06 12:00:00"}}}
```

Schreibt `parse_response(response)`: Ist `status` gleich `"ok"`, gibt die Funktion ein **flaches Dictionary** mit den Schlüsseln `'station'` (Name der Stadt/Station), `'aqi'` und `'time'` zurück. Bei jedem anderen Status gibt sie `None` zurück.

In [ ]:
def parse_response(response: dict):
    """Gibt station, aqi und time als flaches Dictionary zurück, bei einem Fehler None."""
    # TODO
    pass

In [ ]:
responses = [
    {"status": "ok", "data": {"aqi": 42, "city": {"name": "Berlin Mitte"}, "time": {"s": "2026-10-06 12:00:00"}}},
    {"status": "ok", "data": {"aqi": 87, "city": {"name": "Berlin Neukölln"}, "time": {"s": "2026-10-06 12:00:00"}}},
    {"status": "error", "data": "Invalid key"},
    {"status": "ok", "data": {"aqi": 130, "city": {"name": "Berlin Wedding"}, "time": {"s": "2026-10-06 11:00:00"}}},
]
assert parse_response(responses[0]) == {'station': 'Berlin Mitte', 'aqi': 42, 'time': '2026-10-06 12:00:00'}, f"Ergebnis: {parse_response(responses[0])}"
assert parse_response(responses[3])['aqi'] == 130, "Vierte Antwort"
assert parse_response(responses[2]) is None, "Bei status 'error' soll None zurückgegeben werden"
print("✓ Aufgabe D1 bestanden")

### Aufgabe D2: Viele Antworten in einen DataFrame

Schreibt `responses_to_df(responses)`: Die Funktion bekommt eine **Liste von API-Antworten**, wertet jede mit `parse_response` aus und gibt einen **DataFrame** mit den Spalten `station`, `aqi` und `time` zurück, eine Zeile pro **erfolgreicher** Antwort (fehlgeschlagene werden übersprungen). Sind alle Antworten fehlgeschlagen, ist der DataFrame leer, hat aber **trotzdem die drei Spalten**.

In [ ]:
def responses_to_df(responses: list) -> pd.DataFrame:
    """Wertet alle Antworten aus und gibt die erfolgreichen als DataFrame zurück."""
    # TODO
    pass

In [ ]:
responses = [
    {"status": "ok", "data": {"aqi": 42, "city": {"name": "Berlin Mitte"}, "time": {"s": "2026-10-06 12:00:00"}}},
    {"status": "ok", "data": {"aqi": 87, "city": {"name": "Berlin Neukölln"}, "time": {"s": "2026-10-06 12:00:00"}}},
    {"status": "error", "data": "Invalid key"},
    {"status": "ok", "data": {"aqi": 130, "city": {"name": "Berlin Wedding"}, "time": {"s": "2026-10-06 11:00:00"}}},
]
df_air = responses_to_df(responses)
assert isinstance(df_air, pd.DataFrame), "Es soll ein DataFrame zurückgegeben werden"
assert df_air.columns.tolist() == ['station', 'aqi', 'time'], f"Spalten: {df_air.columns.tolist()}"
assert df_air.shape == (3, 3), f"Form: {df_air.shape} (die fehlgeschlagene Antwort wird übersprungen)"
assert df_air['aqi'].tolist() == [42, 87, 130], "Werte der Spalte aqi"
assert df_air['station'].tolist() == ['Berlin Mitte', 'Berlin Neukölln', 'Berlin Wedding'], "Stationen"
empty = responses_to_df([{"status": "error", "data": "x"}])
assert empty.shape == (0, 3) and empty.columns.tolist() == ['station', 'aqi', 'time'], "Nur Fehler: leerer DataFrame mit Spalten"
print("✓ Aufgabe D2 bestanden")

### Aufgabe D3: Kategorien bilden

Schreibt `aqi_category(aqi)`: Die Funktion ordnet einen AQI-Wert einer (vereinfachten) Kategorie zu:

| AQI | Kategorie |
|---|---|
| bis einschließlich 50 | `'gut'` |
| 51 bis einschließlich 100 | `'mäßig'` |
| über 100 | `'schlecht'` |

In [ ]:
def aqi_category(aqi: int) -> str:
    """Ordnet einen AQI-Wert einer Kategorie zu (gut, mäßig, schlecht)."""
    # TODO
    pass

In [ ]:
assert aqi_category(42) == 'gut', "42"
assert aqi_category(87) == 'mäßig', "87"
assert aqi_category(130) == 'schlecht', "130"
assert aqi_category(50) == 'gut' and aqi_category(51) == 'mäßig', "Grenze bei 50"
assert aqi_category(100) == 'mäßig' and aqi_category(101) == 'schlecht', "Grenze bei 100"
assert aqi_category(0) == 'gut', "0"
print("✓ Aufgabe D3 bestanden")

### Aufgabe D4: Kategorie als Spalte

Schreibt `add_category(df)`: Die Funktion gibt eine **Kopie** des DataFrames mit einer zusätzlichen Spalte `kategorie` zurück. Sie enthält für jede Zeile das Ergebnis von `aqi_category` für den Wert in `aqi` (z. B. mit `apply`). Das Original bleibt unverändert.

In [ ]:
def add_category(df: pd.DataFrame) -> pd.DataFrame:
    """Gibt eine Kopie von df mit der Spalte kategorie zurück."""
    # TODO
    pass

In [ ]:
responses = [
    {"status": "ok", "data": {"aqi": 42, "city": {"name": "Berlin Mitte"}, "time": {"s": "2026-10-06 12:00:00"}}},
    {"status": "ok", "data": {"aqi": 87, "city": {"name": "Berlin Neukölln"}, "time": {"s": "2026-10-06 12:00:00"}}},
    {"status": "error", "data": "Invalid key"},
    {"status": "ok", "data": {"aqi": 130, "city": {"name": "Berlin Wedding"}, "time": {"s": "2026-10-06 11:00:00"}}},
]
df_air = responses_to_df(responses)
result = add_category(df_air)
assert result['kategorie'].tolist() == ['gut', 'mäßig', 'schlecht'], f"Kategorien: {result['kategorie'].tolist()}"
assert 'kategorie' not in df_air.columns, "Das Original darf nicht verändert werden (copy() verwenden)"
assert result.columns.tolist() == ['station', 'aqi', 'time', 'kategorie'], f"Spalten: {result.columns.tolist()}"
print("✓ Aufgabe D4 bestanden")

### Aufgabe D5: Bericht

Schreibt `report(df)`: Die Funktion gibt einen **mehrzeiligen String** zurück, mit **einer Zeile pro Station**, sortiert nach `aqi` **absteigend** (schlechteste Luft zuerst), im Format

`Berlin Wedding: AQI 130 (schlecht)`

Die Zeilen werden mit `'\n'` verbunden (keine Leerzeile am Ende). Nutzt dafür `add_category` und einen **f-String**.

In [ ]:
def report(df: pd.DataFrame) -> str:
    """Gibt eine Zeile pro Station zurück, sortiert nach AQI absteigend."""
    # TODO
    pass

In [ ]:
responses = [
    {"status": "ok", "data": {"aqi": 42, "city": {"name": "Berlin Mitte"}, "time": {"s": "2026-10-06 12:00:00"}}},
    {"status": "ok", "data": {"aqi": 87, "city": {"name": "Berlin Neukölln"}, "time": {"s": "2026-10-06 12:00:00"}}},
    {"status": "error", "data": "Invalid key"},
    {"status": "ok", "data": {"aqi": 130, "city": {"name": "Berlin Wedding"}, "time": {"s": "2026-10-06 11:00:00"}}},
]
df_air = responses_to_df(responses)
text = report(df_air)
expected = (
    "Berlin Wedding: AQI 130 (schlecht)\n"
    "Berlin Neukölln: AQI 87 (mäßig)\n"
    "Berlin Mitte: AQI 42 (gut)"
)
assert text == expected, f"Ergebnis:\n{text}"
assert df_air['station'].tolist()[0] == 'Berlin Mitte', "Das Original darf nicht umsortiert werden"
one = pd.DataFrame({'station': ['Berlin Mitte'], 'aqi': [10], 'time': ['x']})
assert report(one) == "Berlin Mitte: AQI 10 (gut)", "Eine Station"
print("✓ Aufgabe D5 bestanden")

**Reflexion:** Ihr habt in Teil C und D gesehen, dass Daten aus dem Web mal als **HTML** (Scraping) und mal als **JSON** (API) ankommen. Wann würdet ihr eine **API** dem Scraping vorziehen? Welche **rechtlichen** und **technischen** Punkte müsst ihr beim Scraping beachten?

*Eure Antwort:* 

### Gesamtbericht

Führt die Zelle aus, sobald alle Schritte bestanden sind.

In [ ]:
responses = [
    {"status": "ok", "data": {"aqi": 42, "city": {"name": "Berlin Mitte"}, "time": {"s": "2026-10-06 12:00:00"}}},
    {"status": "ok", "data": {"aqi": 87, "city": {"name": "Berlin Neukölln"}, "time": {"s": "2026-10-06 12:00:00"}}},
    {"status": "error", "data": "Invalid key"},
    {"status": "ok", "data": {"aqi": 130, "city": {"name": "Berlin Wedding"}, "time": {"s": "2026-10-06 11:00:00"}}},
]
df_air = responses_to_df(responses)
print(report(df_air))
print()
print("Antworten mit Fehler:", sum(parse_response(r) is None for r in responses))
print("Status 404 ->", check_status(404))

---

# Bonus

**Bonus 1: CSV speichern und lesen**

Speichert `df_air` mit `to_csv` (ohne Index, `index=False`) und lest die Datei mit `pd.read_csv` wieder ein. Sind beide DataFrames gleich (`equals`)? Was passiert mit der Spalte `time`?

**Bonus 2: Eine echte Webseite**

Ruft mit `requests.get` eine Webseite eurer Wahl ab (mit `headers` und `timeout`), gebt den Statuscode mit `check_status` aus und lest den Titel (`<title>`) mit Beautiful Soup aus. Was passiert bei einer URL, die es nicht gibt?

**Bonus 3: Mehrere Keys**

Verbindet zwei DataFrames über **zwei** Spalten (`on=['id', 'jahr']`). Was ändert sich im Ergebnis gegenüber einem Join über nur eine Spalte?

**Bonus 4: Eigene kleine API**

Baut mit Flask eine API, die zu `/aqi/<station>` den AQI-Wert aus `df_air` als JSON zurückgibt (Status `404`, wenn die Station unbekannt ist). Das Beispiel `newsapi.py` aus der Vorlesung hilft.

---

# Abschluss

**Zum Schluss:** Startet den Kernel neu (*Kernel → Restart & Run All*). Laufen alle Testzellen **ohne Fehler** durch?

Wichtig für die nächste Woche (**Projekt-Kick-off**): Ihr sucht eine **Projektidee** und einen **Datensatz**. Mit pandas aus dieser Woche prüft ihr dann, ob der Datensatz geeignet ist (`read_csv`, `head`, `info`, `describe`).